# Extract eICU Time Series Data — Respiration & SpO2
Features extracted:
* Respiratory Rate (`respiration`)
* Oxygen Saturation (`sao2` — actually SpO2, see official paper for details)

NaN values are replaced with the **previous recorded value** (forward fill) per patient.

Pre-req: build eICU database and run `sql/patient_id.sql`

In [1]:
import sys
sys.path.append("../")
from helpers.loader import DataBaseLoader
load = DataBaseLoader(user='ansongeorge', password='', dbname='eicu', schema='eicu')

SQL_SCRIPT = """
SELECT
	vitalPeriodic.patientunitstayid,
	observationoffset,
	respiration,
	CASE
		WHEN respiration IS NULL THEN 1
		ELSE 0
	END AS respiration_null,
	sao2,
	CASE
		WHEN sao2 IS NULL THEN 1
		ELSE 0
	END AS sao2_null
 
FROM vitalPeriodic

WHERE vitalPeriodic.patientunitstayid = {id}
AND
(SELECT MIN(observationoffset) FROM vitalPeriodic WHERE patientunitstayid = {id} GROUP BY patientunitstayid) <= {start_time} AND (SELECT MAX(observationoffset) FROM vitalPeriodic WHERE patientunitstayid = {id} GROUP BY patientunitstayid) >= {end_time}
AND
observationoffset <= {end_time} AND observationoffset >= {start_time}

ORDER BY observationoffset
"""

def extract(id, start_time, end_time):
    df = load.query(SQL_SCRIPT.format(id=id, end_time=end_time, start_time=start_time))
    return df

## Get patients and do train test split

In [2]:
unitstay_id = load['patient_id']
unitstay_id = unitstay_id.dropna()          # drop ones without labels, this is caused by the join operation
unitstay_id[['sepsis', 'ami', 'akf', 'heart_failure']] = unitstay_id[['sepsis', 'ami', 'akf', 'heart_failure']].astype(int)
unitstay_id.to_csv('../data/eicu-extract/unitstay_id.csv', index=False)
print(f"There are {len(unitstay_id)} patients in the database.")

There are 170586 patients in the database.


In [3]:
from helpers.utils import seed_everything
from sklearn.model_selection import train_test_split
seed_everything(2023)
X, y = unitstay_id.drop('hospital_expire_flag', axis=1), unitstay_id['hospital_expire_flag']
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=2023, stratify=y)

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/threadpoolctl.py:1214: RuntimeWarning: 
Found Intel OpenMP ('libiomp') and LLVM OpenMP ('libomp') loaded at
the same time. Both libraries are known to be incompatible and this
can cause random crashes or deadlocks on Linux when loaded in the
same Python program.
Using threadpoolctl may cause crashes or deadlocks. For more
information and possible workarounds, please see
    https://github.com/joblib/threadpoolctl/blob/master/multiple_openmp.md

  warnings.warn(msg, RuntimeWarning)


Seed set to 2023



In [4]:
print(f"Mortality rate in training set: {100*y_train.sum() / len(y_train):.2f}%\nMortality rate in testing set: {100*y_test.sum() / len(y_test):.2f}%")

Mortality rate in training set: 9.27%
Mortality rate in testing set: 9.27%


In [5]:
print(f"Training set size: {len(X_train)}\nTesting set size: {len(X_test)}")

Training set size: 127939
Testing set size: 42647


In [6]:
import pandas as pd
train = pd.concat([X_train, y_train], axis=1)
test = pd.concat([X_test, y_test], axis=1)
train.to_csv('../data/eicu-extract/TRAIN-unitstay_id.csv', index=False)
test.to_csv('../data/eicu-extract/TEST-unitstay_id.csv', index=False)

## Extract Time Series

In [7]:
import pandas as pd
train = pd.read_csv('../data/eicu-extract/TRAIN-unitstay_id.csv')
test = pd.read_csv('../data/eicu-extract/TEST-unitstay_id.csv')
train.head()

,patientunitstayid,sepsis,ami,heart_failure,akf,hospital_expire_flag
0,2121063,0,0,0,0,0
1,244478,0,0,0,0,0
2,2935120,0,0,0,0,0
3,3344999,0,0,0,0,0
4,422778,0,0,0,0,0


In [8]:
start_time, end_time, timesteps = 60, 1440, 276

eligible = load.query(f"""
SELECT patientunitstayid
FROM eicu.vitalperiodic
GROUP BY patientunitstayid
HAVING MIN(observationoffset) <= {start_time}
   AND MAX(observationoffset) >= {end_time};
""")

eligible_ids = eligible["patientunitstayid"].astype(int).tolist()
len(eligible_ids)

114962

In [9]:
import numpy as np
from tqdm import tqdm

# Features for this notebook: respiration and sao2
FEATURES = ["respiration", "sao2"]

def fetch_window(ids_chunk, start_time, end_time):
    ids_sql = ",".join(map(str, ids_chunk))
    q = f"""
    SELECT patientunitstayid, observationoffset,
           respiration, sao2
    FROM eicu.vitalperiodic
    WHERE patientunitstayid IN ({ids_sql})
      AND observationoffset BETWEEN {start_time} AND {end_time}
    ORDER BY patientunitstayid, observationoffset;
    """
    return load.query(q)

def forward_fill_per_patient(vitals_df, features):
    """
    Replace NaN values with the previous recorded value (forward fill)
    for each patient independently.
    """
    vitals_df = vitals_df.copy()
    vitals_df[features] = (
        vitals_df.groupby("patientunitstayid")[features]
        .transform(lambda col: col.ffill())
    )
    return vitals_df

def lets_extract_fast(ids, labels_df, start_time, end_time, timesteps, chunk_size=2000):

    label_map = (
        labels_df.set_index("patientunitstayid")["hospital_expire_flag"]
        .dropna()
        .astype(int)
        .to_dict()
    )

    ids = [int(i) for i in ids if int(i) in label_map]

    results = []

    for i in tqdm(range(0, len(ids), chunk_size), desc="Chunks"):
        chunk = ids[i:i+chunk_size]
        vitals = fetch_window(chunk, start_time, end_time)
        if vitals.empty:
            continue

        # Forward fill NaN with previous recorded value per patient
        vitals = forward_fill_per_patient(vitals, FEATURES)

        for pid, g in vitals.groupby("patientunitstayid", sort=False):

            if len(g) < timesteps:
                continue

            off = g["observationoffset"].to_numpy()
            if len(off) < timesteps or not np.all(np.diff(off[:timesteps]) == 5):
                continue

            # (T, 2)
            x = g[FEATURES].to_numpy(dtype=float)[:timesteps, :]

            # convert to (2, T)
            channels = x.T

            results.append(channels)

    return np.asarray(results)

### Training

In [10]:
filtered_train = train[train["patientunitstayid"].isin(eligible_ids)]

result_train = lets_extract_fast(
    eligible_ids,
    labels_df=filtered_train,
    start_time=60,
    end_time=1440,
    timesteps=276,
    chunk_size=2000
)

# Shape: (N_patients, 2, 276)
# Channels: respiration, sao2
result_train.shape

Chunks: 100%|██████████| 41/41 [57:29<00:00, 84.13s/it] 


(46827, 2, 276)

In [11]:
import torch
torch.save(torch.tensor(result_train.astype(np.float64)), '../data/eicu/TRAIN-eicu_respiration_sao2.pt')

### Testing

In [12]:
filtered_test = test[test["patientunitstayid"].isin(eligible_ids)]

result_test = lets_extract_fast(
    eligible_ids,
    labels_df=filtered_test,
    start_time=60,
    end_time=1440,
    timesteps=276,
    chunk_size=2000
)

# Shape: (N_patients, 2, 276)
result_test.shape

Chunks: 100%|██████████| 14/14 [12:38<00:00, 54.19s/it]


(15626, 2, 276)

In [13]:
import torch
torch.save(torch.tensor(result_test.astype(np.float64)), '../data/eicu/TEST-eicu_respiration_sao2.pt')